# 2. Atactic polystyrene: stereocenters survive initialization

Soft DPD repulsion lets atoms pass through each other, so a tetrahedral
center can invert. FlowerMD's stereochemistry guard records the handedness of
every center and holds it with a native dihedral restraint; the run record
audits the centers before DPD, after DPD and after FIRE. Here we also check
them after the unbiased Sage minimization.

Whole chains are placed as rigid units (`unit="chain"`) because a
polystyrene backbone center has a neighbour in the next repeat.

In [1]:
import hoomd

# GPU if one is visible, otherwise CPU. OpenMM follows the same choice.
try:
    DEVICE = hoomd.device.GPU()
    OPENMM_PLATFORM = "CUDA"
except Exception:
    DEVICE = hoomd.device.CPU()
    OPENMM_PLATFORM = "CPU"
print(DEVICE, OPENMM_PLATFORM)

<hoomd.device.GPU object at 0x1466bba26030> CUDA


In [2]:
# The frozen all-atom PhantomWalk protocol, written out explicitly.
PROTOCOL = dict(
    bonded="uff", A=1250.0, gamma=200.0, kT=1.0, r_cut=3.5, bonded_scale=30.0,
    epsilon_weighting=True, protect_stereochemistry=True, stereo_k=30000.0,
)
RUN = dict(
    dpd_min_steps=3500, dpd_chunk=500, dpd_max_steps=40000, energy_tol=0.02,
    consecutive=2, dpd_samples_per_chunk=5, fire_steps=100, fire_dt=0.001,
    require_convergence=False,
)
DT = 0.001

In [3]:
import unyt as u
from flowermd.library import AllAtomDPD, AllAtomLattice, AllAtomPhantomWalk, PolyStyrene
from flowermd.internal.stereochemistry import audit_stereochemistry
from phantomwalk.all_atom import sage_handoff

chains = PolyStyrene(lengths=20, num_mols=8, tacticity="atactic", seed=3)
system = AllAtomLattice(chains, density=1.04 * u.g / u.cm**3, unit="chain", seed=3)
ff = AllAtomDPD(system.system, **PROTOCOL)
print(f"{ff.stereo_centers} stereocenters recorded")
sim = AllAtomPhantomWalk.from_system(system, forcefield=ff, dt=DT, device=DEVICE, seed=3)
record = sim.run_initialization(**RUN)
for stage, audit in record["stereochemistry"].items():
    print(f"{stage:>9}: {audit['n_centers']} centers, inverted {audit['inverted_count']}, "
          f"passed {audit['passed']}")

2026-09-23 15:45:35,306 - mbuild.compound - WARNING - Compound.box.lengths < Compound.boundingbox.lengths. There may be particles outside of the defined simulation box.
152 stereocenters recorded
Initializing simulation state from a gsd.hoomd.Frame.
Step 5500 of 40000; TPS: 0.0; ETA: nan hours, nan minutes
Step 11000 of 40000; TPS: 0.0; ETA: nan hours, nan minutes
Step 16500 of 40000; TPS: 0.0; ETA: nan hours, nan minutes
  initial: 152 centers, inverted 0, passed True
 post_dpd: 152 centers, inverted 0, passed True
post_fire: 152 centers, inverted 0, passed True


In [4]:
import numpy as np

box_a = np.asarray(ff.frame.configuration.box[:3])
handoff, minimized_a = sage_handoff(sim.to_compound(), box_a / 10, platform=OPENMM_PLATFORM)
after = audit_stereochemistry(ff.stereo_reference, minimized_a, box_lengths=box_a,
                              planar_tolerance=ff.stereo_planar_tolerance)
print(f"after Sage minimization: inverted {after['inverted_count']}, passed {after['passed']}; "
      f"energy removed {handoff['energy_removed_sage_epsilon_atom']:.2f} eps_max per atom")

after Sage minimization: inverted 0, passed True; energy removed 3.68 eps_max per atom
